# Apache Arrow Flight

![Arrow Logo](images/arrow.png)


# The Scenario

![City Bikes](images/citybike.jpg)

> Photo by <a href="https://unsplash.com/@jacegrandinetti?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText">Jace & Afsoon</a> on <a href="https://unsplash.com/photos/assorted-color-bicycles-park-beside-blue-rails-near-river-VEXIwDcY1gw?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText">Unsplash</a>

We have our CityBike API, and we need to fetch 1,000,000 records

We need to fetch and process all the records for our analysis.

## With REST API

We have access to a REST API, looks something like this, classic JSON REST API

In [ ]:
import httpx2
import polars as pl


REST_API_URL = "http://rest:8000"

In [ ]:
# Wake up the remote server
req = httpx2.get(f"{REST_API_URL}/health", timeout=10)
req.json()

Let's grab the first few rows, just to see what the data looks like

In [ ]:
req = httpx2.get(
    f"{REST_API_URL}/data/rides/all?num_rows=100",
    headers={"Authorization": "Bearer pydata_amsterdam"},
)
pl.from_records(req.json())

Let's try it again with all the data

In [ ]:
%%timeit -r 1
req = httpx2.get(
    f"{REST_API_URL}/data/rides/all",
    headers={"Authorization": "Bearer pydata_amsterdam"},
)
pl.from_records(req.json())

## With Arrow Flight

Let's try that again, but with an Arrow Flight server instead

In [ ]:
from pyarrow import flight

FLIGHT_SERVER_URL = "grpc://server:7001"

In [ ]:
client = flight.connect(FLIGHT_SERVER_URL)
# Ensure server is responding
client.wait_for_available(5)

In [ ]:
%%timeit -r 1
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))
data = client.do_get(info.endpoints[0].ticket)
pl.from_arrow(data.read_all())